# Stage 1: Sentiment Classifier Training (XLM-RoBERTa)

Trains on SentiTaglish_ProductsAndServices.csv to classify reviews as
negative, neutral, positive, or mixed.

**Before running:** set `SMOKE_TEST = True` in the config cell below and
run everything once. It uses a tiny subset and 1 epoch, just to prove the
pipeline works end to end. Once that finishes with no errors, set
`SMOKE_TEST = False` and run the full thing.

**Also check:** the DATA_PATH in the config cell matches wherever your
uploaded dataset actually landed under /kaggle/input/.

## 1. Setup

In [1]:
!pip install -q -U transformers datasets scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 48.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 51.1 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 53.1 MB/s eta 0:00:00ta 0:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sklearn-compat 0.1.5 requires scikit-learn<1.9,>=1.2, but you have scikit-learn 1.9.0 which is incompatible.


In [2]:
import re
import unicodedata
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from datasets import Dataset

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Torch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


## 2. Config

In [3]:
# --- CHECK THIS PATH ---
# In Kaggle, uploaded datasets live under /kaggle/input/<dataset-name>/<filename>
# Update this to match yourDATA_PATH = "/kaggle/input/datasets/nahokeel/sentimentanalysis/SentiTaglish_ProductsAndServices.csv" actual dataset name once you've added it as an input.
DATA_PATH = "/kaggle/input/datasets/nahokeel/sentimentanalysis/SentiTaglish_ProductsAndServices.csv"

OUTPUT_DIR = "/kaggle/working/stage1_sentiment"

SMOKE_TEST = False          # set False for the real full run
SMOKE_TEST_SIZE = 200      # total rows used across all classes in smoke test mode
EPOCHS = 5                 # used only when SMOKE_TEST = False

SEED = 42
MODEL_NAME = "xlm-roberta-base"
MAX_LENGTH = 128
LEARNING_RATE = 2e-5
BATCH_SIZE = 8

LABEL_NAMES = ["negative", "neutral", "positive", "mixed"]
LABEL2ID = {name: i for i, name in enumerate(LABEL_NAMES)}
ID2LABEL = {i: name for name, i in LABEL2ID.items()}

# Numeric codes in the raw CSV, confirmed against Table 1 in the proposal
SENTIMENT_CODE_MAP = {1: "negative", 2: "neutral", 3: "positive", 4: "mixed"}

torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cpu":
    print("WARNING: no GPU detected. Check Notebook options > Accelerator is set to GPU.")

Using device: cuda


## 3. Preprocessing

In [4]:
SPELLING_MAP = {
    "d2": "dito", "dto": "dito",
    "un": "yun",
    "sya": "siya", "cya": "siya",
    "nde": "hindi", "hnd": "hindi",
    "wla": "wala", "wlang": "walang",
    "eto": "ito",
    "pde": "pwede", "pwd": "pwede",
    "gud": "good",
    "thnx": "thanks", "tnx": "thanks",
    "salamt": "salamat",
}

_REPEATED_CHAR = re.compile(r"(.)\1{2,}")
_WHITESPACE = re.compile(r"\s+")

def clean_text(text: str) -> str:
    if not isinstance(text, str) or not text.strip():
        return ""
    text = unicodedata.normalize("NFKC", text)
    text = text.lower()
    text = _REPEATED_CHAR.sub(r"\1\1", text)
    tokens = text.split()
    tokens = [SPELLING_MAP.get(t, t) for t in tokens]
    text = " ".join(tokens)
    text = _WHITESPACE.sub(" ", text).strip()
    return text

# quick check
print(clean_text("SOBRAAAA ganda ng product!!! salamt seller"))

sobraa ganda ng product!! salamat seller


## 4. Load and prepare data

In [5]:
df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df)} rows")

df["sentiment_label"] = df["sentiment"].map(SENTIMENT_CODE_MAP)
unmapped = df["sentiment_label"].isna().sum()
if unmapped:
    raise ValueError(f"{unmapped} rows have a sentiment code not in SENTIMENT_CODE_MAP")

df["label"] = df["sentiment_label"].map(LABEL2ID)
df["review_clean"] = df["review"].apply(clean_text)
df = df[df["review_clean"] != ""].reset_index(drop=True)

print(df["sentiment_label"].value_counts())

if SMOKE_TEST:
    df = df.groupby("label", group_keys=False).apply(
        lambda g: g.sample(min(len(g), max(1, SMOKE_TEST_SIZE // 4)), random_state=SEED)
    ).reset_index(drop=True)
    epochs = 1
    print(f"\n[SMOKE TEST] Using {len(df)} rows, 1 epoch")
else:
    epochs = EPOCHS

train_df, val_df = train_test_split(
    df, test_size=0.2, stratify=df["label"], random_state=SEED
)
print(f"\nTrain: {len(train_df)} | Val: {len(val_df)}")

Loaded 10510 rows
sentiment_label
positive    3443
negative    3408
mixed       3397
neutral      262
Name: count, dtype: int64

Train: 8408 | Val: 2102


## 5. Tokenize

In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABEL_NAMES),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

def tokenize_fn(batch):
    return tokenizer(
        batch["review_clean"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )

train_ds = Dataset.from_pandas(train_df[["review_clean", "label"]].reset_index(drop=True))
val_ds = Dataset.from_pandas(val_df[["review_clean", "label"]].reset_index(drop=True))

train_ds = train_ds.map(tokenize_fn, batched=True)
val_ds = val_ds.map(tokenize_fn, batched=True)

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/8408 [00:00<?, ? examples/s]

Map:   0%|          | 0/2102 [00:00<?, ? examples/s]

## 6. Train

In [7]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, preds)
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    precision_micro, recall_micro, f1_micro, _ = precision_recall_fscore_support(
        labels, preds, average="micro", zero_division=0
    )

    return {
        "accuracy": accuracy,
        "macro_f1": f1_macro,
        "macro_precision": precision_macro,
        "macro_recall": recall_macro,
        "micro_f1": f1_micro,
        "micro_precision": precision_micro,
        "micro_recall": recall_micro,
    }

# transformers renamed evaluation_strategy -> eval_strategy in v4.46+.
# Try the new name first, fall back to the old one.
common_training_kwargs = dict(
    output_dir=OUTPUT_DIR,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=epochs,
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    logging_steps=20,
    seed=SEED,
    report_to="none",
)
try:
    training_args = TrainingArguments(eval_strategy="epoch", **common_training_kwargs)
except TypeError:
    training_args = TrainingArguments(evaluation_strategy="epoch", **common_training_kwargs)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Macro Precision,Macro Recall,Micro F1,Micro Precision,Micro Recall
1,0.996830,0.972110,0.815414,0.620026,0.614790,0.626900,0.815414,0.815414,0.815414
2,0.777639,0.931206,0.820647,0.642142,0.706211,0.639868,0.820647,0.820647,0.820647
3,0.795651,0.967798,0.831113,0.687635,0.732031,0.674466,0.831113,0.831113,0.831113
4,0.608487,1.141745,0.838249,0.717662,0.770644,0.697832,0.838249,0.838249,0.838249
5,0.413323,1.228874,0.835871,0.704994,0.736259,0.691426,0.835871,0.835871,0.835871


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2630, training_loss=0.7835195926659008, metrics={'train_runtime': 1350.5158, 'train_samples_per_second': 31.129, 'train_steps_per_second': 1.947, 'total_flos': 2765346848808960.0, 'train_loss': 0.7835195926659008, 'epoch': 5.0})

## 7. Evaluate and save

In [8]:
metrics = trainer.evaluate()
print("Final validation metrics:")
for k, v in metrics.items():
    print(f"  {k}: {v}")

if not SMOKE_TEST:
    trainer.save_model(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)
    print(f"\nModel saved to {OUTPUT_DIR}")
    print("Download it from the Kaggle 'Output' tab on the right after this session ends.")
else:
    print("\n[SMOKE TEST] Model not saved. Set SMOKE_TEST = False above and re-run for the real training run.")

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Training Loss,Validation Loss,Epoch,Accuracy,Macro F1,Macro Precision,Macro Recall,Micro F1,Micro Precision,Micro Recall
0.413323,1.141745,5,0.838249,0.717662,0.770644,0.697832,0.838249,0.838249,0.838249


Final validation metrics:
  eval_loss: 1.1417450904846191
  eval_accuracy: 0.8382492863939106
  eval_macro_f1: 0.7176624993175437
  eval_macro_precision: 0.7706441731262148
  eval_macro_recall: 0.6978315323418887
  eval_micro_f1: 0.8382492863939106
  eval_micro_precision: 0.8382492863939106
  eval_micro_recall: 0.8382492863939106


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model saved to /kaggle/working/stage1_sentiment
Download it from the Kaggle 'Output' tab on the right after this session ends.


In [ ]:
import shutil
shutil.make_archive("/kaggle/working/stage1_sentiment_zip", "zip", "/kaggle/working/stage1_sentiment")
print("Zipped to /kaggle/working/stage1_sentiment_zip.zip")